# 27.07 - Monthly checkpoint

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** July CV postmortem based on a compact image baseline and an offline source-to-target transfer experiment.

This checkpoint uses a harder synthetic domain shift. You will build either a small integrated Vision Transformer or a standard TorchVision preset with `weights=None`, train locally on a source domain, and transfer the locally learned encoder to a target domain.


## Core Ideas

- `weights=None` means random initialization. Replacing its head alone is architectural adaptation, not transfer learning.
- Genuine offline transfer is possible: train on a local source task, copy the learned encoder, replace or reset the head, then fine-tune on the target task.
- The full `vit_b_16(weights=None)` architecture is available offline but is too large for this tiny CPU exercise.
- The notebook therefore trains a small integrated `torchvision.models.VisionTransformer` configuration while retaining an optional `vit_b_16` branch for architecture practice.
- Compare scratch and transferred models with the same target split, seed, optimizer, and epoch budget.


In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import VisionTransformer, vit_b_16, resnet18
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

SEED = 27
NUM_CLASSES = 3
IMAGE_SIZE = 32
np.random.seed(SEED)
torch.manual_seed(SEED)


## Prepared Harder Domain-Shift Data

Classes are defined by vertical, horizontal, or diagonal structures. The target domain adds stronger noise, occlusion, position shifts, and a color permutation.

**Return structure — `make_domain_shift_images`:** Returns a `dict` with `source_images` and `target_images`, CPU `torch.float32` tensors shaped `[N_source, 3, S, S]` and `[N_target, 3, S, S]` with values clipped to `[0,1]`; and `source_labels` and `target_labels`, CPU `torch.long` tensors shaped `[N_source]` and `[N_target]`. `S=image_size`, `N_source=source_per_class*3`, and `N_target=target_per_class*3`.


In [2]:
def make_domain_shift_images(source_per_class=15, target_per_class=12, image_size=IMAGE_SIZE, seed=SEED):
    rng = np.random.default_rng(seed)
    source_images, source_labels = [], []
    target_images, target_labels = [], []
    for class_index in range(NUM_CLASSES):
        for harder, sample_count in ((False, source_per_class), (True, target_per_class)):
            for _ in range(sample_count):
                noise_scale = 0.18 if harder else 0.07
                image = rng.normal(0.16, noise_scale, size=(3, image_size, image_size)).astype(np.float32)
                shift = int(rng.integers(-4, 5)) if harder else int(rng.integers(-1, 2))
                width = 2 if harder else 3
                color = [class_index, (class_index + 1) % 3] if harder else [class_index]
                center = image_size // 2 + shift
                if class_index == 0:
                    left = max(1, center - width)
                    right = min(image_size - 1, center + width + 1)
                    image[color, 3:image_size - 3, left:right] += 0.82
                elif class_index == 1:
                    top = max(1, center - width)
                    bottom = min(image_size - 1, center + width + 1)
                    image[color, top:bottom, 3:image_size - 3] += 0.82
                else:
                    for row in range(4, image_size - 4):
                        col = row + shift
                        if 2 <= col < image_size - 2:
                            image[color, row, col - width:col + width + 1] += 0.82
                if harder:
                    occlusion_row = int(rng.integers(8, image_size - 8))
                    occlusion_col = int(rng.integers(8, image_size - 8))
                    image[:, occlusion_row:occlusion_row + 5, occlusion_col:occlusion_col + 6] *= 0.15
                    target_images.append(np.clip(image, 0.0, 1.0))
                    target_labels.append(class_index)
                else:
                    source_images.append(np.clip(image, 0.0, 1.0))
                    source_labels.append(class_index)
    source_order = rng.permutation(len(source_labels))
    target_order = rng.permutation(len(target_labels))
    return {
        "source_images": torch.tensor(np.stack(source_images)[source_order], dtype=torch.float32),
        "source_labels": torch.tensor(np.asarray(source_labels)[source_order], dtype=torch.long),
        "target_images": torch.tensor(np.stack(target_images)[target_order], dtype=torch.float32),
        "target_labels": torch.tensor(np.asarray(target_labels)[target_order], dtype=torch.long),
    }


domains = make_domain_shift_images()
print("source:", domains["source_images"].shape, torch.bincount(domains["source_labels"]).tolist())
print("target:", domains["target_images"].shape, torch.bincount(domains["target_labels"]).tolist())


source: torch.Size([45, 3, 32, 32]) [15, 15, 15]
target: torch.Size([36, 3, 32, 32]) [12, 12, 12]


## Exercise 27-A: Select an offline architecture

Implement three architecture choices. `tiny_vit` must use the integrated `VisionTransformer` class; `vit_b_16` and `resnet18` must explicitly use `weights=None` and replace their classification heads. The smoke check uses `tiny_vit` to remain fast.

**Return structure — `build_offline_model`:** Returns an `nn.Module` on CPU. For every supported architecture its forward pass accepts a `torch.float32` tensor `[N,3,H,W]` and returns a `torch.float32` logits tensor `[N,num_classes]`. Supported strings are exactly `tiny_vit`, `vit_b_16`, and `resnet18`.


In [3]:
# TODO 27-A
def build_offline_model(num_classes, architecture="tiny_vit"):
    if architecture == "tiny_vit" : 
        model = VisionTransformer(
            image_size = IMAGE_SIZE,
            patch_size = 2,
            num_classes = num_classes,
            num_layers=2,
            num_heads=2,
            hidden_dim=32,
            mlp_dim=64,
            dropout=0.05,
            )
    elif architecture == "vit_b_16" : 
        model = vit_b_16(weights = None)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    else :
        model = resnet18(weights = None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model
    

# Smoke check: run this after implementing the function above.
smoke_model = build_offline_model(NUM_CLASSES, architecture="tiny_vit")
smoke_logits = smoke_model(domains["source_images"][:2])
print("offline model logits:", smoke_logits.shape)


offline model logits: torch.Size([2, 3])


## Exercise 27-B: Train a classifier

Use Adam, cross-entropy, `model.train()`, and mean loss weighted by batch size.

**Return structure — `train_classifier`:** Returns a `list[float]` of length `epochs`. Each item is the finite non-negative mean training loss for that epoch. The supplied model is updated in place and remains on CPU.


In [6]:
# TODO 27-B
def train_classifier(model, loader, epochs=3, learning_rate=0.003):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr = learning_rate
    )
    device = "cpu"
    model.train()

    losses = []

    for _ in range(epochs) : 
        total_loss = 0
        total_samples = 0
        for images, labels in loader :
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits,labels)

            loss.backward()
            optimizer.step()

            total_samples += len(images)
            total_loss += loss.item()
        losses.append(total_loss/total_samples)
    return losses
            
        


# Smoke check: run this after implementing the function above.
smoke_loader = DataLoader(TensorDataset(domains["source_images"][:18], domains["source_labels"][:18]), batch_size=9, shuffle=True)
smoke_history = train_classifier(smoke_model, smoke_loader, epochs=1, learning_rate=0.003)
print("one-epoch history:", smoke_history)


one-epoch history: [0.12068235874176025]


## Exercise 27-C: Evaluate consistently

Use `model.eval()` and `torch.inference_mode()`, then calculate accuracy and Macro-F1.

**Return structure — `evaluate_classifier`:** Returns a `dict` with exactly `logits`, a CPU `torch.float32` tensor `[N,C]`; `labels` and `predictions`, CPU `torch.long` tensors `[N]`; and `accuracy` and `macro_f1`, Python `float` values in `[0,1]`.


In [9]:
# TODO 27-C
def evaluate_classifier(model, loader):
    model.eval()
    all_logits = []
    all_preds = []
    all_labels = []
    with torch.inference_mode() : 
        for images, labels in loader : 
            logits = model(images)
            all_logits.append(logits)
            all_labels.append(labels)
            preds = logits.argmax(dim = 1)
            all_preds.append(preds)
        all_logits = torch.cat(all_logits, dim = 0)
        all_preds = torch.cat(all_preds, dim = 0)
        all_labels = torch.cat(all_labels, dim = 0)
        accuracy = accuracy_score(all_labels,all_preds)
        macro_f1 = f1_score(all_labels,all_preds,average = 'macro')
    return {
        "logits" : all_logits,
        "labels" : all_labels,
        "predictions" : all_preds,
        "accuracy" : accuracy,
        "macro_f1" : macro_f1
    }

            


# Smoke check: run this after implementing the function above.
smoke_metrics = evaluate_classifier(smoke_model, smoke_loader)
print("smoke Macro-F1:", smoke_metrics["macro_f1"])


smoke Macro-F1: 0.20512820512820515


## Exercise 27-D: Transfer locally learned features

Create a fresh tiny ViT, load the source model state, reset the classification head, and optionally freeze the encoder. This is genuine transfer because the source model has learned local source-domain features.

**Return structure — `make_transfer_model`:** Returns a tuple. Position 0 is a CPU `VisionTransformer` with an output head for `num_classes`. Position 1 is a `dict` with `frozen_parameter_count` and `trainable_parameter_count`, both positive or zero Python `int` values. When `freeze_encoder=True`, only parameters under `heads` are trainable.


In [12]:
# TODO 27-D
def make_transfer_model(source_model, num_classes, freeze_encoder=False):
    target_model = build_offline_model(num_classes)
    target_model.load_state_dict(source_model.state_dict())
    torch.nn.init.zeros_(target_model.heads.head.bias)
    torch.nn.init.normal_(target_model.heads.head.weight, mean = 0.0, std = 0.02)
    if freeze_encoder : 
        for name, params in target_model.named_parameters() : 
            params.requires_grad = name.startswith("head")
    frozen_parameter_count = sum(params.numel() for params in target_model.parameters() if not params.requires_grad)
    trainable_parameter_count = sum(params.numel() for params in target_model.parameters() if params.requires_grad)
    return (target_model,{
        "frozen_parameter_count" : frozen_parameter_count,
        "trainable_parameter_count" : trainable_parameter_count
    })



# Smoke check: run this after implementing the function above.
smoke_transfer, smoke_transfer_report = make_transfer_model(smoke_model, NUM_CLASSES, freeze_encoder=True)
print("transfer parameter report:", smoke_transfer_report)


transfer parameter report: {'frozen_parameter_count': 25824, 'trainable_parameter_count': 99}


## Exercise 27-E: Run the checkpoint comparison

Pretrain one tiny ViT on the source domain. On the same target split, compare a randomly initialized target model against a model initialized from the locally learned source state.

**Return structure — `run_checkpoint_comparison`:** Returns a `dict` with exactly `source_history`, `scratch_history`, and `transfer_history` (`list[float]`, each with its requested epoch length); `scratch_metrics` and `transfer_metrics` (the dictionaries returned by `evaluate_classifier`); and `target_split`, a dictionary of CPU tensors `train_indices` and `val_indices`, both `torch.long` and disjoint.


In [ ]:
# TODO 27-E
def run_checkpoint_comparison(domains, source_epochs=3, target_epochs=3, seed=SEED):
    torch.manual_seed(seed)
    source_loader = DataLoader(
        TensorDataset(domains["source_images"],domains["source_labels"]),
        batch_size = 15,
        shuffle = True,
        generator = torch.Generator().manual_seed(seed)
    )
    all_indices = np.arange(len(domains["target_labels"]))
    train_array, test_array = train_test_split(
        all_indices,
        test_size = 0.25,
        random_state = seed,
        stratify = domains["target_labels"].numpy()
    )
    train_indices = torch.tensor(train_array)
    val_indices = torch.tensor(test_array)
    train_loader = DataLoader(
        TensorDataset(domains["target_images"][train_indices],domains["target_labels"][train_indices]),
        batch_size = 15,
        shuffle = True,
        generator = torch.Generator().manual_seed(seed)
    )
    val_loader = DataLoader(
        TensorDataset(domains["target_images"][val_indices],domains["target_labels"][val_indices]),
        batch_size = 15,
        shuffle = True,
        generator = torch.Generator().manual_seed(seed)
    )
    num_classes = len(set(domains["source_labels"]))
    baseline_model = build_offline_model(num_classes)
    source_history = train_classifier(baseline_model,source_loader,source_epochs)
    scratch_model = build_offline_model(num_classes)
    scratch_history = train_classifier(scratch_model,train_loader,target_epochs)
    transfer_model = make_transfer_model(scratch_model,num_classes,True)[0]
    transfer_history = train_classifier(transfer_model,train_loader,target_epochs)
    scratch_metrics = evaluate_classifier(scratch_model,val_loader)
    transfer_metrics = evaluate_classifier(transfer_model,val_loader)
    return{
        "source_history" : source_history,
        "scratch_history" : scratch_history,
        "transfer_history" : transfer_history,
        "scratch_metrics" : scratch_metrics,
        "transfer_metrics" : transfer_metrics,
        "target_split" : {
            "train_indices" : train_indices,
            "val_indices" : val_indices
        }
    }


# Smoke check: run this after implementing the function above.
checkpoint_result = run_checkpoint_comparison(domains, source_epochs=2, target_epochs=2, seed=SEED)
print("scratch/transfer Macro-F1:", checkpoint_result["scratch_metrics"]["macro_f1"], checkpoint_result["transfer_metrics"]["macro_f1"])


scratch/transfer Macro-F1: 0.16666666666666666 0.16666666666666666


## Test Cases

Run this cell after completing all TODO cells. A correct implementation prints `Day 27 tests passed`.

**Return structure — `run_day27_tests`:** Returns `None`. Success is communicated by assertions completing and the exact printed message `Day 27 tests passed`.


In [18]:
def run_day27_tests():
    tiny_model = build_offline_model(NUM_CLASSES, architecture="tiny_vit")
    output = tiny_model(torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE))
    assert output.shape == (2, NUM_CLASSES) and output.dtype == torch.float32
    assert sum(parameter.numel() for parameter in tiny_model.parameters()) < 200000

    loader = DataLoader(TensorDataset(domains["source_images"][:12], domains["source_labels"][:12]), batch_size=6)
    history = train_classifier(tiny_model, loader, epochs=1, learning_rate=0.002)
    assert len(history) == 1 and isinstance(history[0], float) and np.isfinite(history[0])
    metrics = evaluate_classifier(tiny_model, loader)
    assert set(metrics) == {"logits", "labels", "predictions", "accuracy", "macro_f1"}
    assert metrics["logits"].shape == (12, NUM_CLASSES)

    transferred, report = make_transfer_model(tiny_model, NUM_CLASSES, freeze_encoder=True)
    assert isinstance(transferred, VisionTransformer)
    assert report["frozen_parameter_count"] > 0 and report["trainable_parameter_count"] > 0
    assert all(parameter.requires_grad == name.startswith("heads") for name, parameter in transferred.named_parameters())

    result = checkpoint_result
    assert set(result) == {"source_history", "scratch_history", "transfer_history", "scratch_metrics", "transfer_metrics", "target_split"}
    assert len(result["source_history"]) == 2 and len(result["scratch_history"]) == 2
    train_set = set(result["target_split"]["train_indices"].tolist())
    val_set = set(result["target_split"]["val_indices"].tolist())
    assert train_set.isdisjoint(val_set)
    print("Day 27 tests passed")


run_day27_tests()


Day 27 tests passed


## Day 27 Checklist

- [ ] I can distinguish random initialization from transfer learning.
- [ ] I used an integrated offline transformer architecture without downloading weights.
- [ ] I locally pretrained on a source domain before transferring to the target.
- [ ] I compared scratch and transfer using the same target split and budget.
- [ ] I wrote three concrete August improvement actions.
